# Qwen2.5-3B direct GRPO

This notebook uses the completed dataset and starts GRPO directly from the base model; no SFT run is required.

In [ ]:
import os, subprocess, sys
from pathlib import Path
LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv>=1,<2'], check=True)
from dotenv import load_dotenv
ENV_FILE = Path(os.environ.get('CRASHDIAG_ENV_FILE', LAUNCH_DIR / 'env.txt')).expanduser()
if not ENV_FILE.is_absolute(): ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if ENV_FILE.is_file(): load_dotenv(ENV_FILE, override=True)
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None
KAGGLE_SECRET_ALIASES = {'HF_TOKEN': 'HF_TOKEN', 'CRASHDIAG_DATASET_RUN_ID': 'CRASHDIAG_DATASET_RUN_ID', 'DATASET_RUN_ID': 'CRASHDIAG_DATASET_RUN_ID', 'CRASHDIAG_SANDBOX_URL': 'CRASHDIAG_SANDBOX_URL', 'CRASHDIAG_API_TOKEN': 'CRASHDIAG_API_TOKEN', 'CRASHDIAG_SANDBOX_TOKEN': 'CRASHDIAG_SANDBOX_TOKEN', 'CRASHDIAG_SOURCE_COMMIT': 'CRASHDIAG_SOURCE_COMMIT', 'SOURCE_COMMIT': 'CRASHDIAG_SOURCE_COMMIT'}
loaded_kaggle_secrets, kaggle_secret_errors = [], {}
if UserSecretsClient is not None:
    client = UserSecretsClient()
    for secret_name, env_name in KAGGLE_SECRET_ALIASES.items():
        if os.environ.get(env_name): continue
        try: value = client.get_secret(secret_name)
        except Exception as exc: kaggle_secret_errors[secret_name] = f'{type(exc).__name__}: {exc}'; continue
        if value: os.environ[env_name] = value; loaded_kaggle_secrets.append(secret_name)
print('loaded Kaggle secret names:', loaded_kaggle_secrets or 'none')
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
REPO_URL = os.environ.get('CRASHDIAG_REPO_URL', 'https://github.com/Indium-AI-Labs/CrashDiag.git')
SOURCE_COMMIT = os.environ.get('CRASHDIAG_SOURCE_COMMIT', 'main')
WORKDIR = Path(os.environ.get('CRASHDIAG_WORKDIR', LAUNCH_DIR / 'CrashDiag-runtime')).expanduser().resolve()
if (WORKDIR / '.git').is_dir(): subprocess.run(['git', '-C', str(WORKDIR), 'fetch', 'origin', 'main'], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()): raise RuntimeError(f'CRASHDIAG_WORKDIR is not a Git checkout: {WORKDIR}')
else: subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
subprocess.run(['git', '-C', str(WORKDIR), 'checkout', SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'bitsandbytes'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train]'], check=True)
print('env_file=', ENV_FILE if ENV_FILE.is_file() else 'not present (runtime/Kaggle secrets)')


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
MODEL_SLUG = "qwen2.5_3b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get('CRASHDIAG_DATASET_RUN_ID', '').strip()
if not DATASET_RUN_ID: raise RuntimeError('Set CRASHDIAG_DATASET_RUN_ID to the completed dataset run ID.')
def ist_run_id(stage): return datetime.now(ZoneInfo('Asia/Kolkata')).strftime('%Y%m%dT%H%M%SIST') + f'-{MODEL_SLUG}-{stage}'
GRPO_RUN_ID = os.environ.get('CRASHDIAG_GRPO_RUN_ID', '').strip() or ist_run_id('grpo')
GRPO_EVAL_RUN_ID = os.environ.get('CRASHDIAG_GRPO_EVAL_RUN_ID', '').strip() or ist_run_id('grpo-eval')
CURRICULUM = os.environ.get('CRASHDIAG_CURRICULUM', 'v5').strip().lower()
TRAIN_FILE, EVAL_FILE = 'grpo_train.jsonl', 'grpo_eval.jsonl'
SANDBOX_TOKEN = os.environ.get('CRASHDIAG_API_TOKEN') or os.environ.get('CRASHDIAG_SANDBOX_TOKEN', '')
print(f'base_model={BASE_MODEL}, curriculum={CURRICULUM}, dataset_run_id={DATASET_RUN_ID}')
print(f'GRPO_RUN_ID={GRPO_RUN_ID}, GRPO_EVAL_RUN_ID={GRPO_EVAL_RUN_ID}')


In [ ]:
from training.artifacts import ArtifactConfig, ArtifactUploader
DATASET_DIR = Path('artifacts/datasets')
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ['HF_TOKEN'])).download_stage('datasets', DATASET_DIR)
assert (DATASET_DIR / TRAIN_FILE).is_file() and (DATASET_DIR / EVAL_FILE).is_file()
print('dataset ready:', DATASET_DIR)


In [ ]:
import subprocess, sys
print('=== DIRECT GRPO FROM BASE ===', flush=True)
command = [sys.executable, '-m', 'accelerate.commands.launch', '--num_processes', os.environ.get('CRASHDIAG_GRPO_NUM_PROCESSES', '2'), '--num_machines', '1', '--mixed_precision', 'fp16', '--dynamo_backend', 'no', '-m', 'training.grpo', '--model', BASE_MODEL, '--train-file', str(DATASET_DIR / TRAIN_FILE), '--eval-file', str(DATASET_DIR / EVAL_FILE), '--output-dir', 'outputs/grpo', '--no-load-in-4bit', '--precision', 'fp16', '--batch-size', '2', '--gradient-accumulation-steps', '2', '--num-generations', os.environ.get('CRASHDIAG_GRPO_NUM_GENERATIONS', '4'), '--learning-rate', os.environ.get('CRASHDIAG_GRPO_LEARNING_RATE', '5e-6'), '--lr-scheduler-type', os.environ.get('CRASHDIAG_GRPO_LR_SCHEDULER', 'constant_with_warmup'), '--warmup-ratio', os.environ.get('CRASHDIAG_GRPO_WARMUP_RATIO', '0.05'), '--temperature', os.environ.get('CRASHDIAG_GRPO_TEMPERATURE', '1.0'), '--top-p', os.environ.get('CRASHDIAG_GRPO_TOP_P', '0.95'), '--logging-steps', os.environ.get('CRASHDIAG_GRPO_LOGGING_STEPS', '10'), '--save-steps', os.environ.get('CRASHDIAG_GRPO_SAVE_STEPS', '200'), '--max-prompt-length', '1024', '--max-completion-length', os.environ.get('CRASHDIAG_GRPO_MAX_COMPLETION_LENGTH', '96'), '--max-steps', os.environ.get('CRASHDIAG_GRPO_MAX_STEPS', '832'), '--eval-steps', '1000000000', '--artifact-bucket', BUCKET_ID, '--run-id', GRPO_RUN_ID, '--artifact-stage', 'grpo', '--sandbox-url', os.environ['CRASHDIAG_SANDBOX_URL'], '--sandbox-token', SANDBOX_TOKEN]
subprocess.run(command, check=True)
print('GRPO complete:', GRPO_RUN_ID)


In [ ]:
from training.evaluate_jsonl import main as evaluate_main
print('=== GRPO EVALUATION ===', flush=True)
exit_code = evaluate_main(['--model', 'outputs/grpo', '--dataset', str(DATASET_DIR / EVAL_FILE), '--output-dir', 'outputs/grpo-eval', '--load-in-4bit', '--precision', 'bf16', '--max-new-tokens', '64', '--sandbox-url', os.environ['CRASHDIAG_SANDBOX_URL'], '--sandbox-token', SANDBOX_TOKEN, '--artifact-bucket', BUCKET_ID, '--run-id', GRPO_EVAL_RUN_ID, '--artifact-stage', 'grpo-eval', '--no-few-shot'])
if exit_code: raise RuntimeError(f'GRPO evaluation failed: {exit_code}')
print('GRPO evaluation complete:', GRPO_EVAL_RUN_ID)


In [ ]:
from IPython.display import SVG, display
REPORTS_DIR = Path('outputs/grpo-eval') / 'reports'
charts = sorted(REPORTS_DIR.glob('*.svg'))
if not charts: raise RuntimeError(f'No SVG charts were generated in {REPORTS_DIR}')
print(f'Uploaded reports: hf://buckets/{BUCKET_ID}/runs/{GRPO_EVAL_RUN_ID}/grpo-eval/reports')
for chart in charts: display(SVG(filename=str(chart)))
